# Power BI report fix: the numbers

The source of truth for every number the reports must show, and for the before and after timings. Client values come only from `config/client.yaml` (`load_config()`); the input is the flat export in `data/input/`.

1. Check the config and the flat export, and load its mapped columns under standard names.
2. Compute, with plain SQL, every number each card, chart and table must show. The slow report and the fixed report must both match these.
3. Write `output/sales.csv`, the standard file the fixed report reads.
4. Read the Power BI measurements in `data/input/` (`measure.py`) once they are there.
5. Write `powerbi/06-checks.md`, the check numbers in the Power BI build files, and `output/numbers.json`.

Run: `jupyter nbconvert --to notebook --execute --inplace analysis/analysis.ipynb`.

In [1]:
import json
import re
import sys
from pathlib import Path

import duckdb
import pandas as pd

sys.path.insert(0, "..")
from config import REPORT_COLUMNS, check_inputs, load_config
from measure import read_measurements

cfg = load_config()
check_inputs(cfg)
ROOT = Path("..")
FLAT = cfg["input_dir"] / cfg["inputs"]["flat"]
Y = cfg["report"]["check_year"]
SECOND = cfg["report"]["second_check"]
pd.set_option("display.float_format", "{:,.2f}".format)

con = duckdb.connect()
# The mapped columns only, under their standard names; extra columns in the export are ignored.
mapped = ", ".join(f'"{header}" AS {name}' for name, header in cfg["columns"].items())
con.sql(f"CREATE TABLE sales AS SELECT {mapped} FROM read_csv('{FLAT.as_posix()}')")
q = lambda sql, params=None: con.execute(sql, params or []).df()

flat_columns = len(con.sql(f"SELECT * FROM read_csv('{FLAT.as_posix()}') LIMIT 0").columns)
overview = q('''
SELECT count(*)                       AS sales_rows,
       count(DISTINCT order_key)      AS orders,
       count(DISTINCT customer_key)   AS customers,
       count(DISTINCT product_key)    AS products,
       min(order_date)                AS first_order,
       max(order_date)                AS last_order,
       round(sum(quantity * net_price)) AS sales_all_years
FROM sales
''')
overview

,sales_rows,orders,customers,products,first_order,last_order,sales_all_years
0,2098633,875901,88063,2517,2015-01-01,2024-04-20,"2,127,928,962.00"


## What the report must show

The same measures as the report, written in SQL. Values are rounded the way the report shows them.

| Measure | Rule |
|---|---|
| Sales Amount | sum of Quantity × Net Price |
| Margin % | (Sales Amount − sum of Quantity × Unit Cost) ÷ Sales Amount |
| Orders, Customers | distinct order keys, distinct customer keys |
| Avg Order Value | Sales Amount ÷ Orders |
| Sales PY | Sales Amount for the same months one year earlier, with every other filter kept |
| Sales YoY % | (Sales Amount − Sales PY) ÷ Sales PY |
| Product Rank | rank of the product's Sales Amount among the products in the current filters, ties share a rank |

`kpis()` returns the six cards for one slicer selection: `report.check_year` with nothing else selected, and `report.second_check`.

In [2]:
def kpis(year, label, country=None, category=None):
    where, params = "", []
    for column, value in (("country", country), ("category", category)):
        if value is not None:
            where += f" AND {column} = ?"
            params.append(value)
    cards = q(f'''
    WITH now AS (
        SELECT sum(quantity * net_price)                 AS sales,
               sum(quantity * (net_price - unit_cost))   AS margin,
               count(DISTINCT order_key)                 AS orders,
               count(DISTINCT customer_key)              AS customers
        FROM sales WHERE year(order_date) = {year}{where}
    ), py AS (
        SELECT sum(quantity * net_price) AS sales_py FROM sales WHERE year(order_date) = {year - 1}{where}
    )
    SELECT round(sales)                                   AS "Sales Amount",
           round(100 * (sales - sales_py) / sales_py, 1)  AS "Sales YoY %",
           round(100 * margin / sales, 1)                 AS "Margin %",
           orders                                         AS "Orders",
           round(sales / orders, 2)                       AS "Avg Order Value",
           customers                                      AS "Customers",
           round(sales_py)                                AS "Sales PY"
    FROM now, py
    ''', params * 2)
    if pd.isna(cards.iloc[0]["Sales Amount"]):
        raise SystemExit(f"{label} matches no rows in data/input/{FLAT.name}")
    return cards

cards_default = kpis(Y, "report.check_year")
cards_default

,Sales Amount,Sales YoY %,Margin %,Orders,Avg Order Value,Customers,Sales PY
0,"318,425,878.00",-28.40,56.00,159695,"1,993.96",62337,"444,516,719.00"


In [3]:
cards_second = kpis(SECOND["year"], "report.second_check", SECOND["country"], SECOND["category"])
cards_second

,Sales Amount,Sales YoY %,Margin %,Orders,Avg Order Value,Customers,Sales PY
0,"19,645,197.00",93.50,56.00,9854,"1,993.63",5197,"10,153,937.00"


In [4]:
# Page 1 chart: Sales Amount and Sales PY by month of the check year.
by_month = q(f'''
SELECT strftime(min(order_date), '%b') AS "Month",
       round(sum(quantity * net_price) FILTER (WHERE year(order_date) = {Y}))     AS "Sales Amount",
       round(sum(quantity * net_price) FILTER (WHERE year(order_date) = {Y - 1})) AS "Sales PY"
FROM sales
WHERE year(order_date) IN ({Y - 1}, {Y})
GROUP BY month(order_date) ORDER BY month(order_date)
''')
by_month

,Month,Sales Amount,Sales PY
0,Jan,"34,197,837.00","36,052,336.00"
1,Feb,"40,800,178.00","49,140,854.00"
2,Mar,"22,604,720.00","29,700,003.00"
3,Apr,"11,939,474.00","19,221,969.00"
4,May,"28,979,019.00","44,587,592.00"
5,Jun,"27,454,645.00","46,105,321.00"
6,Jul,"23,332,124.00","34,978,428.00"
7,Aug,"24,849,603.00","35,682,325.00"
8,Sep,"25,110,286.00","36,801,749.00"
9,Oct,"25,116,723.00","36,552,588.00"


In [5]:
# Page 1 bars: Sales Amount by Category and by Country in the check year.
by_category = q(f'''
SELECT category AS "Category", round(sum(quantity * net_price)) AS "Sales Amount"
FROM sales WHERE year(order_date) = {Y} GROUP BY 1 ORDER BY 2 DESC
''')
by_country = q(f'''
SELECT country AS "Country", round(sum(quantity * net_price)) AS "Sales Amount"
FROM sales WHERE year(order_date) = {Y} GROUP BY 1 ORDER BY 2 DESC
''')
display(by_category, by_country)

,Category,Sales Amount
0,Computers,"113,112,204.00"
1,Cell phones,"59,754,615.00"
2,Home Appliances,"54,115,079.00"
3,TV and Video,"41,323,827.00"
4,"Music, Movies and Audio Books","21,768,661.00"
5,Cameras and camcorders,"18,765,677.00"
6,Audio,"6,847,569.00"
7,Games and Toys,"2,738,246.00"


,Country,Sales Amount
0,United States,"156,132,838.00"
1,Germany,"41,279,015.00"
2,Canada,"37,241,921.00"
3,United Kingdom,"26,252,830.00"
4,Australia,"22,522,330.00"
5,Netherlands,"17,031,456.00"
6,France,"11,734,079.00"
7,Italy,"6,231,408.00"


In [6]:
# Page 2 table: every product sold in the check year by Sales Amount; the top 10 go in the checks.
products = q(f'''
WITH p AS (
    SELECT product_name,
           sum(quantity * net_price) FILTER (WHERE year(order_date) = {Y})                     AS sales,
           sum(quantity * net_price) FILTER (WHERE year(order_date) = {Y - 1})                 AS sales_py,
           sum(quantity * (net_price - unit_cost)) FILTER (WHERE year(order_date) = {Y})       AS margin,
           count(DISTINCT customer_key) FILTER (WHERE year(order_date) = {Y})                  AS customers
    FROM sales WHERE year(order_date) IN ({Y - 1}, {Y})
    GROUP BY 1
)
SELECT rank() OVER (ORDER BY sales DESC)             AS "Product Rank",
       product_name                                  AS "Product Name",
       round(sales)                                  AS "Sales Amount",
       round(sales_py)                               AS "Sales PY",
       round(100 * (sales - sales_py) / sales_py, 1) AS "Sales YoY %",
       round(100 * margin / sales, 1)                AS "Margin %",
       customers                                     AS "Customers"
FROM p WHERE sales IS NOT NULL
ORDER BY sales DESC
''')
print(len(products), "products in the table")
products.head(10)

2517 products in the table


,Product Rank,Product Name,Sales Amount,Sales PY,Sales YoY %,Margin %,Customers
0,1,"Adventure Works 52"" LCD HDTV X590 White","2,317,817.00","2,712,426.00",-14.50,64.80,262
1,2,"Adventure Works 52"" LCD HDTV X590 Brown","2,264,428.00","2,867,452.00",-21.00,64.70,270
2,3,"Adventure Works 52"" LCD HDTV X590 Silver","2,038,838.00","3,218,644.00",-36.70,65.00,228
3,4,"Adventure Works 52"" LCD HDTV X590 Black","1,804,751.00","3,159,420.00",-42.90,64.30,207
4,5,Adventure Works Desktop PC2.33 XD233 White,"1,803,958.00","3,444,403.00",-47.60,64.80,621
5,6,Adventure Works Desktop PC2.33 XD233 Brown,"1,775,896.00","3,110,374.00",-42.90,64.60,623
6,7,WWI Desktop PC2.33 X2330 Silver,"1,733,712.00","3,152,473.00",-45.00,65.00,607
7,8,Adventure Works Desktop PC2.33 XD233 Silver,"1,712,446.00","3,158,969.00",-45.80,64.90,634
8,9,WWI Desktop PC2.33 X2330 Black,"1,678,976.00","3,077,120.00",-45.40,64.90,627
9,10,WWI Desktop PC2.33 X2330 Brown,"1,657,784.00","3,108,136.00",-46.70,64.80,615


In [7]:
# Page 2 matrix: Category rows with the grand total, check year.
category_matrix = q(f'''
SELECT coalesce(category, 'Total') AS "Category",
       round(sum(quantity * net_price) FILTER (WHERE year(order_date) = {Y})) AS "Sales Amount",
       round(100 * (sum(quantity * net_price) FILTER (WHERE year(order_date) = {Y})
                  - sum(quantity * net_price) FILTER (WHERE year(order_date) = {Y - 1}))
                  / sum(quantity * net_price) FILTER (WHERE year(order_date) = {Y - 1}), 1) AS "Sales YoY %",
       round(100 * sum(quantity * (net_price - unit_cost)) FILTER (WHERE year(order_date) = {Y})
                 / sum(quantity * net_price) FILTER (WHERE year(order_date) = {Y}), 1) AS "Margin %"
FROM sales WHERE year(order_date) IN ({Y - 1}, {Y})
GROUP BY ROLLUP (category)
ORDER BY category IS NULL, 2 DESC
''')
category_matrix

,Category,Sales Amount,Sales YoY %,Margin %
0,Computers,"113,112,204.00",-36.10,56.20
1,Cell phones,"59,754,615.00",-27.70,54.00
2,Home Appliances,"54,115,079.00",-16.40,55.30
3,TV and Video,"41,323,827.00",-28.10,56.90
4,"Music, Movies and Audio Books","21,768,661.00",-22.30,58.60
5,Cameras and camcorders,"18,765,677.00",-21.10,58.20
6,Audio,"6,847,569.00",-10.40,55.10
7,Games and Toys,"2,738,246.00",-13.20,52.00
8,Total,"318,425,878.00",-28.40,56.00


In [8]:
# The demo's slow Sales PY takes the year from the rows in each cell, so it matches the fixed report only where
# every cell shown has sales in the selected year. Count the checked cells that have none; 06-checks.md reports it.
empty_cells = q(f'''
SELECT
  (SELECT 12 - count(DISTINCT month(order_date)) FROM sales WHERE year(order_date) = {Y}) AS months_without_sales,
  (SELECT 12 - count(DISTINCT month(order_date)) FROM sales
    WHERE year(order_date) = {SECOND["year"]} AND country = ? AND category = ?)            AS months_without_sales_second_check,
  (SELECT count(DISTINCT subcategory) FROM sales WHERE year(order_date) IN ({Y - 1}, {Y}))
    - (SELECT count(DISTINCT subcategory) FROM sales WHERE year(order_date) = {Y})        AS subcategories_without_sales,
  (SELECT count(*) FROM (SELECT DISTINCT product_name FROM sales WHERE year(order_date) = {Y - 1}
                         EXCEPT SELECT DISTINCT product_name FROM sales WHERE year(order_date) = {Y})) AS products_sold_only_the_year_before
''', [SECOND["country"], SECOND["category"]])
empty_cells

,months_without_sales,months_without_sales_second_check,subcategories_without_sales,products_sold_only_the_year_before
0,0,0,0,0


In [9]:
# The standard file the fixed report reads (powerbi/01-power-query.md): the mapped columns under the report's names.
cfg["output_dir"].mkdir(exist_ok=True)
renamed = ", ".join(f'{name} AS "{REPORT_COLUMNS[name]}"' for name in REPORT_COLUMNS)
con.sql(f"COPY (SELECT {renamed} FROM sales) TO '{(cfg['output_dir'] / 'sales.csv').as_posix()}' (HEADER, DELIMITER ',')")
print("wrote output/sales.csv")

wrote output/sales.csv


## Before and after: timings and model size

Measured in Power BI Desktop with the method in `docs/measuring.md`: one Performance Analyzer export per page per report (`before-<page>.json`, `after-<page>.json`) and one VertiPaq Analyzer export per report (`before.vpax`, `after.vpax`), all in `data/input/`. Nothing is shown until they are there.

In [10]:
measured = read_measurements(cfg)  # None until the exports are in data/input/
if measured is None:
    print("No measurements in data/input/ yet (docs/measuring.md).")
else:
    pages = cfg["measurements"]["pages"]
    page_table = pd.DataFrame([{"page": p,
                                "before_s": measured["before"]["pages"][p][0],
                                "after_s": measured["after"]["pages"][p][0]} for p in pages])
    page_table["times_faster"] = page_table.before_s / page_table.after_s
    model_table = pd.DataFrame({stage: measured[stage]["model"] for stage in ("before", "after")})
    display(page_table, model_table)

No measurements in data/input/ yet (docs/measuring.md).


## Write the numbers

`powerbi/06-checks.md` is written in full. In the other Power BI build files each number sits between `<!--n:name-->` and `<!--/n-->` markers, and the value from `numbers` replaces whatever is between them. `output/numbers.json` keeps every number for anything else that reports them.

In [11]:
CUR = cfg["client"]["currency"]
money = lambda v: "blank" if pd.isna(v) else f"{CUR}{v:,.0f}"
money2 = lambda v: "blank" if pd.isna(v) else f"{CUR}{v:,.2f}"
pct = lambda v: "blank" if pd.isna(v) else f"{v:.1f}%"
whole = lambda v: f"{int(v):,}"
FORMATS = {"Sales Amount": money, "Sales PY": money, "Sales YoY %": pct, "Margin %": pct,
           "Orders": whole, "Customers": whole, "Avg Order Value": money2}

def md_table(df):
    lines = ["| " + " | ".join(df.columns) + " |", "|" + "---|" * len(df.columns)]
    for _, row in df.iterrows():
        lines.append("| " + " | ".join(FORMATS.get(c, str)(row[c]) for c in df.columns) + " |")
    return "\n".join(lines)

def cards_table(cards):
    row = cards.iloc[0]
    names = ["Sales Amount", "Sales YoY %", "Margin %", "Orders", "Avg Order Value", "Customers"]
    return md_table(pd.DataFrame({"Card": names, "Shows": [FORMATS[n](row[n]) for n in names]}))

o = overview.iloc[0]
d = cards_default.iloc[0]
f = cards_second.iloc[0]
top_cat = category_matrix.iloc[0]  # the biggest category (sorted by Sales Amount, Total last)
START, END = cfg["report"]["date_start"], cfg["report"]["date_end"]  # the Power BI Date table's fixed range
if not START <= o.first_order.date() <= o.last_order.date() <= END:
    raise SystemExit(f"order dates {o.first_order:%Y-%m-%d} to {o.last_order:%Y-%m-%d} fall outside report.date_start "
                     f"to report.date_end in config/client.yaml, the Power BI Date table's range: widen it")
date_rows = (END - START).days + 1
model_rows = pd.DataFrame({
    "Table": ["Sales (both reports)", "Customer (fixed report)", "Product (fixed report)", "Date (fixed report)"],
    "Rows (Table view, bottom left)": [whole(o.sales_rows), whole(o.customers), whole(o.products), whole(date_rows)],
})
e = empty_cells.iloc[0]
empty = int(e.sum())
second = f"Year {SECOND['year']}, Country {SECOND['country']}, Category {SECOND['category']}"

checks = f'''# 6. Checks: the numbers each page must show ({cfg["client"]["name"]})

Written by [analysis/analysis.ipynb](../analysis/analysis.ipynb) from SQL on `data/input/{FLAT.name}`; do not edit by hand. **Both reports, the slow one and the fixed one, must show exactly these numbers.** If one differs, the build has a mistake; fix it before taking any timing.

Checked cells with no sales in the selected year: {empty} (months {int(e.months_without_sales)}, months in the second check {int(e.months_without_sales_second_check)}, subcategories {int(e.subcategories_without_sales)}, products sold only the year before {int(e.products_sold_only_the_year_before)}). The demo's slow `Sales PY` takes the year from the rows in each cell, so it matches the fixed report only where this count is 0.

Small rounding note: a value ending in exactly .5 can round one unit differently in Power BI than in Python. Anything else is a real difference.

## Model

{md_table(model_rows)}

## Page 1: Overview, default state (Year {Y}, all countries, all categories)

{cards_table(cards_default)}

Sales YoY % compares with Sales PY = {money(d["Sales PY"])} ({Y - 1}).

**Sales and previous year by month** (hover a column to see both values):

{md_table(by_month)}

**Sales by category** (bar order, top to bottom):

{md_table(by_category)}

**Sales by country** (bar order, top to bottom):

{md_table(by_country)}

## Page 1: Overview, filtered ({second})

Set these three slicers, check the cards, then set the slicers back to the default state.

{cards_table(cards_second)}

## Page 2: Products, default state (Year {Y})

**Products by sales:** the table has {whole(len(products))} rows (every product sold in {Y}). The top 10:

{md_table(products.head(10))}

**Sales by category** matrix, collapsed to Category:

{md_table(category_matrix)}
'''
(ROOT / "powerbi" / "06-checks.md").write_text(checks, encoding="utf-8")

numbers = {
    "sales_rows": whole(o.sales_rows),
    "rows_million": f"{o.sales_rows / 1e6:.1f} million",
    "orders": whole(o.orders),
    "first_month": o.first_order.strftime("%B %Y"),
    "last_month": o.last_order.strftime("%B %Y"),
    "flat_columns": str(flat_columns),
    "flat_mb": f"{FLAT.stat().st_size / 1e6:,.0f} MB",
    "sales_all": money(o.sales_all_years),
    "customers_dim": whole(o.customers),
    "products_count": whole(o.products),
    "date_rows": whole(date_rows),
    "check_year": str(Y),
    "s_year": str(SECOND["year"]),
    "s_country": SECOND["country"],
    "s_category": SECOND["category"],
    "sales_check": money(d["Sales Amount"]),
    "yoy_check": pct(d["Sales YoY %"]),
    "yoy_check_abs": pct(abs(d["Sales YoY %"])),
    "margin_check": pct(d["Margin %"]),
    "orders_check": whole(d["Orders"]),
    "aov_check": money2(d["Avg Order Value"]),
    "customers_check": whole(d["Customers"]),
    "products_check": whole(len(products)),
    "f_sales": money(f["Sales Amount"]),
    "f_yoy": pct(f["Sales YoY %"]),
    "f_margin": pct(f["Margin %"]),
    "f_orders": whole(f["Orders"]),
    "f_aov": money2(f["Avg Order Value"]),
    "f_customers": whole(f["Customers"]),
    "top_product": products.iloc[0]["Product Name"],
    "top_product_sales": money(products.iloc[0]["Sales Amount"]),
    "top_category": top_cat["Category"],
    "top_category_sales": money(top_cat["Sales Amount"]),
    "top_category_yoy": pct(top_cat["Sales YoY %"]),
    "top_category_margin": pct(top_cat["Margin %"]),
}
if measured:
    seconds = lambda v: f"{v:.1f} s"
    numbers["pages"] = ",".join(cfg["measurements"]["pages"])
    for p in cfg["measurements"]["pages"]:
        before, after = measured["before"]["pages"][p][0], measured["after"]["pages"][p][0]
        numbers[f"{p}_before"], numbers[f"{p}_after"] = seconds(before), seconds(after)
        numbers[f"{p}_change"] = f"{before / after:.1f}× faster"
    slowest = max(cfg["measurements"]["pages"], key=lambda p: measured["before"]["pages"][p][0])
    numbers["slowest_page"] = slowest
    numbers["slowest_before"], numbers["slowest_after"] = numbers[f"{slowest}_before"], numbers[f"{slowest}_after"]
    b, a = measured["before"]["model"], measured["after"]["model"]
    numbers["size_before"], numbers["size_after"] = f"{b['size_mb']:,.0f} MB", f"{a['size_mb']:,.0f} MB"
    numbers["size_down"] = f"{100 * (1 - a['size_mb'] / b['size_mb']):.0f}%"
    for stage, stats in (("before", b), ("after", a)):
        for key in ("tables", "columns", "calculated_columns", "hidden_date_tables"):
            numbers[f"{key}_{stage}"] = str(stats[key])
(cfg["output_dir"] / "numbers.json").write_text(json.dumps(numbers, indent=2, ensure_ascii=False), encoding="utf-8")

def fill(path):
    text = path.read_text(encoding="utf-8")
    text = re.sub(r"<!--n:(\w+)-->(.*?)<!--/n-->",
                  lambda m: f"<!--n:{m[1]}-->{numbers.get(m[1], m[2])}<!--/n-->", text, flags=re.S)
    path.write_text(text, encoding="utf-8")

for name in ["01-power-query.md", "02-model.md", "04-pages.md", "08-build-checklist.md"]:
    fill(ROOT / "powerbi" / name)
numbers

{'sales_rows': '2,098,633',
 'rows_million': '2.1 million',
 'orders': '875,901',
 'first_month': 'January 2015',
 'last_month': 'April 2024',
 'flat_columns': '35',
 'flat_mb': '801 MB',
 'sales_all': '$2,127,928,962',
 'customers_dim': '88,063',
 'products_count': '2,517',
 'date_rows': '3,653',
 'check_year': '2023',
 's_year': '2022',
 's_country': 'Germany',
 's_category': 'Computers',
 'sales_check': '$318,425,878',
 'yoy_check': '-28.4%',
 'yoy_check_abs': '28.4%',
 'margin_check': '56.0%',
 'orders_check': '159,695',
 'aov_check': '$1,993.96',
 'customers_check': '62,337',
 'products_check': '2,517',
 'f_sales': '$19,645,197',
 'f_yoy': '93.5%',
 'f_margin': '56.0%',
 'f_orders': '9,854',
 'f_aov': '$1,993.63',
 'f_customers': '5,197',
 'top_product': 'Adventure Works 52" LCD HDTV X590 White',
 'top_product_sales': '$2,317,817',
 'top_category': 'Computers',
 'top_category_sales': '$113,112,204',
 'top_category_yoy': '-36.1%',
 'top_category_margin': '56.2%'}